# New artifact

This notebook lives on the lab's own disk and is gone when the lab scales down. Declaring with `commit=True` keeps it: the last saved version is copied to the declared artifact's folder as `declaring.ipynb`. (This cell will be replaced with the artifact path.)

In [2]:
import lab
from artifacts.core.artifact import Resources
from artifacts.core.SGD.lr_schedule import WarmupCosine
from artifacts.core.SGD.optimizers import AdamW
from artifacts.core.SGD.training import LoopConfig, TrainingParameters
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import SourceURL
from artifacts.stages.pretraining import Pretraining
from artifacts.stages.pretraining.evaluations import TokenLoss
from artifacts.stages.pretraining.samplers import RandomWindows
from artifacts.tokenizers.bpe import Tokenizer as BPETokenizer
from models.transformer import Transformer
from artifacts.profiles import Profile
from profiling.profile_events import launch_chain
from profiling.util import run_profile

In [ ]:
# a Project Gutenberg mirror: a book is <the digits of its id but the last>/<id>/<id>-0.txt
PGLAF = "https://gutenberg.pglaf.org"
mobydick = SourceURL(name="mobydick", url=f"{PGLAF}/2/7/0/2701/2701-0.txt")
montecristo = SourceURL(name="montecristo", url=f"{PGLAF}/1/1/8/1184/1184-0.txt")
prideprejudice = SourceURL(name="prideprejudice", url=f"{PGLAF}/1/3/4/1342/1342-0.txt")
frankenstein = SourceURL(name="frankenstein", url=f"{PGLAF}/8/84/84-0.txt")
taleoftwocities = SourceURL(name="taleoftwocities", url=f"{PGLAF}/9/98/98-0.txt")
romeojuliet = SourceURL(name="romeojuliet", url=f"{PGLAF}/1/5/1/1513/1513-0.txt")
books = (mobydick, montecristo, prideprejudice, frankenstein, taleoftwocities)

tokenizer = BPETokenizer(
    vocab_size=1024,
    special_tokens=("<|endoftext|>",),
    sources=(mobydick,montecristo,),
    allocated_resources=Resources(cpu=4),
)

dataset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=books,
    valid_sources=(romeojuliet,),
)

pretraining = Pretraining(
    run_id="demo",
    dataset=dataset,
    sampler=RandomWindows(),
    autocast="torch.bfloat16",  # None on a T4, which has no bf16 unit
    model=Transformer(
        vocab_size=tokenizer.vocab_size,
        d_model=128,
        d_ff=384,
        sequence_length=128,
        num_layers=4,
        num_heads=4,
        rope_theta=10000,
        tie_embeddings=False,
        dtype="torch.float32",
    ),
    training_parameters=TrainingParameters(
        total_steps=2000,
        batch_size=64,
        max_norm=1.0,
        lr_schedule=WarmupCosine(
            max_learning_rate=2e-3,
            min_learning_rate=2e-4,
            warmup_iters=100,
            cosine_cycle_iters=2000,
        ),
        optimizer=AdamW(
            betas=(0.9, 0.95),
            weight_decay=0.1,
            eps=1e-8,
        ),
        seed=0,
    ),
    loop_config=LoopConfig(
        checkpoint_every=500,
        evaluations=(TokenLoss(eval_every=100, batch_size=64),),
        gpu_check_every=100,
        compile="reduce-overhead",
    ),
    allocated_resources=Resources(gpu_type="A10", gpu_count=1),
    tags={"experiment": ["demo"]},
)

In [4]:
pretraining.pprint()

Pretraining(
│   run_id='demo',
│   training_parameters=TrainingParameters(
│   │   total_steps=2000,
│   │   batch_size=64,
│   │   max_norm=1.0,
│   │   lr_schedule=WarmupCosine(
│   │   │   max_learning_rate=0.002,
│   │   │   min_learning_rate=0.0002,
│   │   │   warmup_iters=100,
│   │   │   cosine_cycle_iters=2000
│   │   ),
│   │   optimizer=AdamW(
│   │   │   betas=(
│   │   │   │   0.9,
│   │   │   │   0.95
│   │   │   ),
│   │   │   weight_decay=0.1,
│   │   │   eps=1e-08
│   │   ),
│   │   seed=0
│   ),
│   loop_config=LoopConfig(
│   │   evaluations=(
│   │   │   TokenLoss(
│   │   │   │   eval_every=100,
│   │   │   │   batch_size=64
│   │   │   ),
│   │   ),
│   │   gpu_check_every=100,
│   │   checkpoint_every=500,
│   │   optimizer_checkpoint_policy='latest',
│   │   compile='reduce-overhead'
│   ),
│   model=Transformer(
│   │   vocab_size=4096,
│   │   sequence_length=128,
│   │   dtype='torch.float32',
│   │   num_layers=4,
│   │   d_model=128,
│   │   d_ff=384,
│   │   num_heads=4,
│   │   rope_theta=10000,
│   │   tie_embeddings=False
│   ),
│   starting_checkpoint=None,
│   dataset=MappedDataSet(
│   │   train_set=(
│   │   │   TokenizedSource(
│   │   │   │   tokenizer=Tokenizer(
│   │   │   │   │   vocab_size=4096,
│   │   │   │   │   special_tokens=(
│   │   │   │   │   │   '<|endoftext|>',
│   │   │   │   │   ),
│   │   │   │   │   sources=frozenset({
│   │   │   │   │   │   SourceURL(
│   │   │   │   │   │   │   name='mobydick',
│   │   │   │   │   │   │   url='https://gutenberg.pglaf.org/2/7/0/2701/2701-0.txt'
│   │   │   │   │   │   ),
│   │   │   │   │   │   SourceURL(
│   │   │   │   │   │   │   name='montecristo',
│   │   │   │   │   │   │   url='https://gutenberg.pglaf.org/1/1/8/1184/1184-0.txt'
│   │   │   │   │   │   )
│   │   │   │   │   })
│   │   │   │   ),
│   │   │   │   source=SourceURL(
│   │   │   │   │   name='mobydick',
│   │   │   │   │   url='https://gutenberg.pglaf.org/2/7/0/2701/2701-0.txt'
│   │   │   │   )
│   │   │   ),
│   │   │   TokenizedSource(
│   │   │   │   tokenizer=Tokenizer(
│   │   │   │   │   vocab_size=4096,
│   │   │   │   │   special_tokens=(
│   │   │   │   │   │   '<|endoftext|>',
│   │   │   │   │   ),
│   │   │   │   │   sources=frozenset({
│   │   │   │   │   │   SourceURL(
│   │   │   │   │   │   │   name='mobydick',
│   │   │   │   │   │   │   url='https://gutenberg.pglaf.org/2/7/0/2701/2701-0.txt'
│   │   │   │   │   │   ),
│   │   │   │   │   │   SourceURL(
│   │   │   │   │   │   │   name='montecristo',
│   │   │   │   │   │   │   url='https://gutenberg.pglaf.org/1/1/8/1184/1184-0.txt'
│   │   │   │   │   │   )
│   │   │   │   │   })
│   │   │   │   ),
│   │   │   │   source=SourceURL(
│   │   │   │   │   name='montecristo',
│   │   │   │   │   url='https://gutenberg.pglaf.org/1/1/8/1184/1184-0.txt'
│   │   │   │   )
│   │   │   ),
│   │   │   TokenizedSource(
│   │   │   │   tokenizer=Tokenizer(
│   │   │   │   │   vocab_size=4096,
│   │   │   │   │   special_tokens=(
│   │   │   │   │   │   '<|endoftext|>',
│   │   │   │   │   ),
│   │   │   │   │   sources=frozenset({
│   │   │   │   │   │   SourceURL(
│   │   │   │   │   │   │   name='mobydick',
│   │   │   │   │   │   │   url='https://gutenberg.pglaf.org/2/7/0/2701/2701-0.txt'
│   │   │   │   │   │   ),
│   │   │   │   │   │   SourceURL(
│   │   │   │   │   │   │   name='montecristo',
│   │   │   │   │   │   │   url='https://gutenberg.pglaf.org/1/1/8/1184/1184-0.txt'
│   │   │   │   │   │   )
│   │   │   │   │   })
│   │   │   │   ),
│   │   │   │   source=SourceURL(
│   │   │   │   │   name='prideprejudice',
│   │   │   │   │   url='https://gutenberg.pglaf.org/1/3/4/1342/1342-0.txt'
│   │   │   │   )
│   │   │   ),
│   │   │   TokenizedSource(
│   │   │   │   tokenizer=Tokenizer(
│   │   │   │   │   vocab_size=4096,
│   │   │   │   │   special_tokens=(
│   │   │   │   │   │   '<|endoftext|>',
│   │   │   │   │   ),
│   │   │   │   │   sources=frozenset({
│   │   │   │   │   │   SourceURL(
│   │ 

## Declare

Preview first: it writes nothing. Save the notebook before committing, since the copy is the file on disk.

In [ ]:
report = lab.declare(pretraining, verbose=True)

In [ ]:
report = lab.declare(pretraining, verbose=True, commit=True)

## Benchmark

In [ ]:
result = run_profile(Profile(
    run_id=pretraining.run_id,
    resources=pretraining.allocated_resources,
    model=pretraining.model,
    optimizer=pretraining.training_parameters.optimizer,
    batch_size=pretraining.training_parameters.batch_size,
    seed=pretraining.training_parameters.seed,
    warmup_steps=5,
    profile_steps=4,
    autocast=pretraining.autocast,
    compile=pretraining.loop_config.compile,
    kernel_events=True,
    module_scopes=True,
    peak_memory=True,
    trace_memory=True,
))

In [ ]:
result.display_steps(steps=[3])

In [ ]:
step = 3
rows = result.events(step)
groups, kernels, unattributed_kernels, flop_sources = result.aggregate_kernels(step, depth=3)
display(groups[["kernel_count", "gpu_time_us", "gpu_span_us", "total_est_flops"]])

In [ ]:
display(kernels[["name", "phase", "module_fqn", "owner_fqn", "gpu_start_us", "duration_us"]])

In [ ]:
kernel_id = kernels.index[0]
display(launch_chain(rows, kernel_id, format=True))